# Estimator Sheet 
This notebook builds the single combined dataset used for quantile regression estimation. It walks a folder of forecast-card CSVs (`project-*.csv`, `poi-*.csv`, `observations-*.csv`, and optional `forecast-*.csv` files), finds every project, combines each project's files into one merged table, and writes everything out to a single `combined_output.csv`. That combined file is the input for the model-fitting notebook.

**How to use this notebook:**
1. Run the "find project folders" cell to confirm every project was located. Set `DATA_ROOT` to the folder containing all your project CSVs.
2. Run the "build the combined dataset" cell. For each project this loads its project, poi, observations, and forecast CSVs, merges them together, and labels each observation as `before` or `after` the project's actual open date. All projects are then stacked into one `master_df`.
3. Run the "spot-check" cell to visually confirm the merge worked as expected.
4. Run the "write out" cell to save `master_df` to `combined_output.csv`.

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Find your project folders

Point `DATA_ROOT` at the folder holding your project data. This walks the entire directory tree underneath it and collects every sub-folder that contains a `project-*.csv` file (each matching folder is treated as one project). The result is a sorted list of folder paths, printed below so you can confirm the right projects were found.

**Update this path to point at your own data folder before continuing.**

In [2]:
def find_project_folders(root_dir):
    """Return every sub-folder of root_dir that contains a project-*.csv file."""
    folders = []
    for dirpath, _dirnames, filenames in os.walk(root_dir):
        if any(re.match(r"project-.*\.csv$", f, re.IGNORECASE) for f in filenames):
            folders.append(dirpath)
    return sorted(folders)

DATA_ROOT = r"c:\Users\hmod225\Documents\Code\forecastcards_ba_data\data"
project_folders = find_project_folders(DATA_ROOT)
project_folders

['c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-0436',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-0507',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9310',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9503',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9506',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9508',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9603',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9605',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9607',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9608',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9702UPDATE',
 'c:\\Users\\hmod225\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9706',
 'c:\\User

## 2. Build the combined dataset

This section does the actual finding, loading, and merging for every project, then stacks the results into one master table used for estimation.

**Loading a project's files** — for a given project folder, finds and reads:
- `project-*.csv` → project-level metadata (one row per project)
- `poi-*.csv` → "points of interest" tied to that project (e.g. intersections/segments)
- `observations-*.csv` → observed traffic/volume records
- `forecast-*.csv` → **optional** — loaded if present, otherwise treated as missing

**Merging a project's files** — performs three sequential merges:
- **project → poi:** tags each POI row with its `project_id`, then merges project metadata onto every POI row (keeping all POIs, broadcasting the single project row across them).
- **before/after labeling:** converts `observation_date` to a real datetime and compares it to the project's `date_open_actual` to create `before_or_after`.
- **(project+poi) → observations:** merges the project/poi table onto the observations, matching on `poi_id` **and** `before_or_after`, keeping every observation row.
- **+ forecast:** if a forecast file exists, merges it in on `poi_id`, suffixing overlapping columns `_observation`/`_forecast`; otherwise renames `start_time`/`end_time` to the `_observation` versions so column naming stays consistent either way.

In [3]:
def load_project_files(folder):
    """Load the project / poi / observations / (optional) forecast csvs for one folder."""
    project_path = glob.glob(os.path.join(folder, "project-*.csv"))[0]
    poi_path = glob.glob(os.path.join(folder, "poi-*.csv"))[0]
    obs_path = glob.glob(os.path.join(folder, "observations-*.csv"))[0]
    forecast_matches = glob.glob(os.path.join(folder, "forecast-*.csv"))

    project_df = pd.read_csv(project_path)
    poi_df = pd.read_csv(poi_path)
    obs_df = pd.read_csv(obs_path)
    forecast_df = pd.read_csv(forecast_matches[0]) if forecast_matches else None

    return project_df, poi_df, obs_df, forecast_df

In [4]:
def process_project_folder(folder):
    project_df, poi_df, obs_df, forecast_df = load_project_files(folder)

    project_id = project_df["project_id"].iloc[0]
    date_open_actual = pd.to_datetime(project_df["date_open_actual"].iloc[0])

    poi_df = poi_df.copy()
    poi_df["project_id"] = project_id

    # STEP 1: project -> poi
    # project_df is the left frame, so its columns land first in the result.
    # how="right" keeps every poi row, broadcasting the single project row onto each.
    project_poi = pd.merge(project_df, poi_df, on="project_id", how="right")

    # derive a sensible before/after label for each observation
    # an observation counts as "before" if it happened before the project's
    # actual open date, and "after" otherwise
    obs_df = obs_df.copy()
    obs_df["observation_date"] = pd.to_datetime(obs_df["observation_date"], errors="coerce")
    obs_df["before_or_after"] = np.where(
        obs_df["observation_date"] < date_open_actual, "before", "after"
    )
    # binary flag: before -> 0, after -> 1
    obs_df["before_or_after_binary"] = (obs_df["before_or_after"] == "after").astype(int)

    # STEP 2: (project + poi) -> observation
    # project_poi is again the LEFT frame, so project/poi columns stay ahead of the
    # observation columns that get appended on the right. how="right" keeps every
    # observation row, attaching the poi snapshot that matches its before/after label.
    merged = pd.merge(
        project_poi, obs_df, on=["poi_id", "before_or_after"], how="right"
    )

    # STEP 3: + forecast (broadcast across every obs of that POI)
    if forecast_df is not None:
        merged = pd.merge(
            merged, forecast_df, on="poi_id", how="left",
            suffixes=("_observation", "_forecast")
        )
    else:
        # rename raw obs-only columns to match the suffixed naming used when a
        # forecast file IS present, so column names stay consistent across projects
        merged = merged.rename(columns={"start_time": "start_time_observation",
                                         "end_time": "end_time_observation"})

    return merged

In [5]:

project_tables = [process_project_folder(folder) for folder in project_folders]

master_df = pd.concat(project_tables, ignore_index=True, sort=False)

print(f"{len(master_df)} total observation rows across {len(project_tables)} project(s)")
master_df.head(10)

1964 total observation rows across 62 project(s)


,project_id,state,county,project_type,description,date_open_planned,date_horizon,date_open_actual,area_type,project_length_miles,poi_id,before_or_after,facility_name,segment_length_miles,speed_limit_mph,lane_width,signals_per_mile,stop_signs_per_mile,through_lanes,left_turn_lanes,right_turn_lanes,express_hov_hot_lanes,shoulder_width,functional_class,obs_id,observation_date,start_time_observation,end_time_observation,observation_variable,observation_value,observation_unit,before_or_after_binary,forecast_id,forecast_date,start_time_forecast,end_time_forecast,forecast_variable,forecast_value,forecast_unit,forecast_type
0,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,before,Lexington Ave.,NaN,55,12,0,0,2,0,0,0,7,minor arterial,40974_1,2011-01-01,0:00:00,24:00:00,volume,16700.0,AADT,0,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline
1,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,before,Lexington Ave.,NaN,55,12,0,0,2,0,0,0,7,minor arterial,40974_2,2016-01-01,0:00:00,24:00:00,volume,17518.0,AADT,0,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline
2,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,before,Lexington Ave.,NaN,55,12,0,0,2,0,0,0,7,minor arterial,40974_3,2017-01-01,0:00:00,24:00:00,volume,17844.0,AADT,0,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline
3,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,after,Lexington Ave.,NaN,55,12,0,0,2,0,1,0,9,minor arterial,40974_4,2018-01-01,0:00:00,24:00:00,volume,17937.0,AADT,1,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline
4,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,after,Lexington Ave.,NaN,55,12,0,0,2,0,1,0,9,minor arterial,40974_5,2019-01-01,0:00:00,24:00:00,volume,15200.0,AADT,1,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline
5,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,after,Lexington Ave.,NaN,55,12,0,0,2,0,1,0,9,minor arterial,40974_6,2020-01-01,0:00:00,24:00:00,volume,11388.0,AADT,1,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline
6,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,after,Lexington Ave.,NaN,55,12,0,0,2,0,1,0,9,minor arterial,40974_7,2021-01-01,0:00:00,24:00:00,volume,13169.0,AADT,1,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline
7,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,after,Lexington Ave.,NaN,55,12,0,0,2,0,1,0,9,minor arterial,40974_8,2022-01-01,0:00:00,24:00:00,volume,13525.0,AADT,1,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline
8,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,after,Lexington Ave.,NaN,55,12,0,0,2,0,1,0,9,minor arterial,40974_9,2023-01-01,0:00:00,24:00:00,volume,12691.0,AADT,1,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline
9,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,4/1/2017,suburban,0.45,40974,after,Lexington Ave.,NaN,55,12,0,0,2,0,1,0,9,minor arterial,40974_10,2024-01-01,0:00:00,24:00:00,volume,13230.0,AADT,1,575,NaN,0:00:00,24:00:00,volume,19783.71252,AADT,Trendline


## 3. Spot-check the merged data

Displays a narrowed view of just the columns most relevant for spot-checking, the join keys, dates, before/after labels, and a few geometric attributes (lane counts, shoulder width), to visually verify the merges worked correctly before writing anything out.

In [6]:
master_df[[
    "project_id", "poi_id", "observation_date", "date_open_actual",
    "before_or_after", "before_or_after_binary",
    "through_lanes", "right_turn_lanes", "shoulder_width"
]]

,project_id,poi_id,observation_date,date_open_actual,before_or_after,before_or_after_binary,through_lanes,right_turn_lanes,shoulder_width
0,F-M-0436,40974,2011-01-01,4/1/2017,before,0,2,0,7
1,F-M-0436,40974,2016-01-01,4/1/2017,before,0,2,0,7
2,F-M-0436,40974,2017-01-01,4/1/2017,before,0,2,0,7
3,F-M-0436,40974,2018-01-01,4/1/2017,after,1,2,1,9
4,F-M-0436,40974,2019-01-01,4/1/2017,after,1,2,1,9
...,...,...,...,...,...,...,...,...,...
1959,TA-M352,10707,2021-01-01,NaN,after,1,unknown,unknown,unknown
1960,TA-M352,10707,2022-01-01,NaN,after,1,unknown,unknown,unknown
1961,TA-M352,10707,2023-01-01,NaN,after,1,unknown,unknown,unknown
1962,TA-M352,10707,2024-01-01,NaN,after,1,unknown,unknown,unknown


## 4. Write out the combined CSV

Saves `master_df` to `combined_output.csv` inside `DATA_ROOT` (the same folder your project subfolders live in, alongside `validation_report.txt`) and confirms how many rows/columns were written. This file is the input for the quantile regression model-fitting notebook.

In [7]:
output_path = os.path.join(DATA_ROOT, "combined_output.csv")
master_df.to_csv(output_path, index=False)
print(f"Wrote {len(master_df)} rows and {len(master_df.columns)} columns to {output_path}")

Wrote 1964 rows and 40 columns to c:\Users\hmod225\Documents\Code\forecastcards_ba_data\data\combined_output.csv
